# Daily Challenge: Multi-Head Attention & Transformer Comparisons

**Dataset:** XNLI — Cross-lingual Natural Language Inference  
**Languages:** 15 languages (English, French, Arabic, Chinese, Hindi, Thai, etc.)  
**Task:** Classify (premise, hypothesis) pairs as: `0=entailment`, `1=neutral`, `2=contradiction`

## 📦 Step 0: Install & Import

In [ ]:
!pip install torch transformers matplotlib seaborn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 📂 Step 1: Load & Explore the Dataset

In [ ]:
# Upload the zip file from the challenge, then extract:
# !unzip 'Basics_of_BERT_and_XLM-RoBERTa_-_PyTorch_-_2.zip'
# !unzip 'Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip'
# !unzip 'Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip'

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

label_names = {0: "entailment", 1: "neutral", 2: "contradiction"}

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nLanguages ({train_df['language'].nunique()}): {list(train_df['language'].unique())}")
print(f"\nLabel distribution:")
print(train_df['label'].map(label_names).value_counts())

In [ ]:
# Show a few examples across languages
for _, row in train_df.groupby('language').first().head(4).iterrows():
    print(f"[{row.name}]")
    print(f"  Premise:    {row['premise'][:80]}")
    print(f"  Hypothesis: {row['hypothesis'][:80]}")
    print(f"  Label:      {label_names[row['label']]}")
    print()

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Label counts
train_df['label'].map(label_names).value_counts().plot(
    kind='bar', ax=axes[0], color=['#4CAF50','#2196F3','#F44336'], edgecolor='black'
)
axes[0].set_title("Label Distribution")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].tick_params(rotation=0)

# Samples per language
train_df['language'].value_counts().plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='black'
)
axes[1].set_title("Samples per Language")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

---
## 🔧 Part 1: Single-Head Attention

**How it works:**
1. Each token is projected into Q (Query), K (Key), V (Value)
2. Score = softmax(Q × Kᵀ / √d_k) — how much each token should attend to others
3. Output = Score × V — weighted mix of values

In [ ]:
class SingleHeadAttention(nn.Module):
    """
    Scaled Dot-Product Single-Head Attention.
    
    Args:
        hidden_dim (int): size of input token embeddings
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)  # output projection
    
    def forward(self, x, mask=None):
        """
        Args:
            x:    (batch, seq_len, hidden_dim)
            mask: (batch, seq_len) optional padding mask
        Returns:
            output:       (batch, seq_len, hidden_dim)
            attn_weights: (batch, seq_len, seq_len)
        """
        # Step 1: Project to Q, K, V
        Q = self.W_q(x)   # (batch, seq, hidden)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Step 2: Compute scaled dot-product scores
        d_k = self.hidden_dim
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        # scores: (batch, seq_len, seq_len)
        
        # Step 3: Apply padding mask if provided
        if mask is not None:
            # mask=0 means padding token -> set to -inf before softmax
            mask = mask.unsqueeze(1)  # (batch, 1, seq_len)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Step 4: Softmax -> attention weights (probabilities)
        attn_weights = torch.softmax(scores, dim=-1)
        
        # Step 5: Weighted sum of Values
        output = torch.matmul(attn_weights, V)   # (batch, seq_len, hidden)
        output = self.W_o(output)
        
        return output, attn_weights


# ✅ Shape validation
batch, seq_len, hidden_dim = 2, 10, 64
dummy = torch.randn(batch, seq_len, hidden_dim)
sha = SingleHeadAttention(hidden_dim)
out, weights = sha(dummy)

print("=== Single-Head Attention ===")
print(f"Input:   {dummy.shape}")
print(f"Output:  {out.shape}")
print(f"Weights: {weights.shape}  (each row sums to 1)")
print(f"Row sum check: {weights[0][0].sum().item():.4f}  ✅")

---
## 🔧 Part 2: Multi-Head Attention

**Why multiple heads?**  
Each head learns a *different type of relationship* between tokens (syntax, semantics, coreference…).  
They run in parallel on smaller subspaces, then their outputs are concatenated.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention.
    
    Args:
        hidden_dim (int): total embedding size (must be divisible by num_heads)
        num_heads  (int): number of parallel attention heads
        dropout   (float): dropout rate on attention weights
    """
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, \
            f"hidden_dim ({hidden_dim}) must be divisible by num_heads ({num_heads})"
        
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads  # each head works on smaller space
        
        # Single matrices project for all heads at once (efficient)
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)
        
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, x):
        """(batch, seq, hidden) → (batch, heads, seq, head_dim)"""
        batch, seq, _ = x.shape
        x = x.reshape(batch, seq, self.num_heads, self.head_dim)
        return x.transpose(1, 2)  # bring heads dim forward
    
    def combine_heads(self, x):
        """(batch, heads, seq, head_dim) → (batch, seq, hidden)"""
        batch, _, seq, _ = x.shape
        x = x.transpose(1, 2)                      # (batch, seq, heads, head_dim)
        return x.reshape(batch, seq, self.hidden_dim)  # concatenate heads
    
    def forward(self, x, mask=None):
        """
        Args:
            x:    (batch, seq_len, hidden_dim)
            mask: (batch, seq_len) optional padding mask
        Returns:
            output:       (batch, seq_len, hidden_dim)
            attn_weights: (batch, num_heads, seq_len, seq_len)
        """
        # Project and split into heads
        Q = self.split_heads(self.W_q(x))   # (batch, heads, seq, head_dim)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        
        # Scaled dot-product attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = torch.softmax(scores, dim=-1)   # (batch, heads, seq, seq)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum of Values
        context = torch.matmul(attn_weights, V)        # (batch, heads, seq, head_dim)
        
        # Combine heads back and project
        output = self.W_o(self.combine_heads(context)) # (batch, seq, hidden_dim)
        
        return output, attn_weights


# ✅ Shape validation
hidden_dim, num_heads = 128, 4
dummy = torch.randn(2, 10, hidden_dim)
mha = MultiHeadAttention(hidden_dim, num_heads, dropout=0.1)
out, weights = mha(dummy)

print("=== Multi-Head Attention ===")
print(f"Input:         {dummy.shape}")
print(f"Output:        {out.shape}")
print(f"Attn weights:  {weights.shape}  (batch, heads, seq, seq)")
print(f"head_dim:      {mha.head_dim}  (hidden_dim / num_heads = {hidden_dim}/{num_heads})")

---
## 🔧 Part 3: Full Encoder Block + Classifier

A full Transformer encoder block = **MultiHeadAttention → Add&Norm → FeedForward → Add&Norm**

In [ ]:
class FeedForward(nn.Module):
    """Two-layer feedforward network with ReLU activation."""
    def __init__(self, hidden_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim)
        )
    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    """
    Single Transformer Encoder Block.
    Sub-layer 1: MultiHeadAttention + residual + LayerNorm
    Sub-layer 2: FeedForward      + residual + LayerNorm
    """
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.ff    = FeedForward(hidden_dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop  = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Sub-layer 1: attention + residual
        attn_out, attn_weights = self.attn(x, mask)
        x = self.norm1(x + self.drop(attn_out))   # Add & Norm
        
        # Sub-layer 2: feedforward + residual
        x = self.norm2(x + self.drop(self.ff(x))) # Add & Norm
        
        return x, attn_weights


class CustomTransformerNLI(nn.Module):
    """
    Custom Transformer Encoder for NLI classification.
    Architecture:
      Token Embedding + Positional Embedding
      → N × EncoderBlock
      → CLS token → Linear classifier
    """
    def __init__(self, vocab_size, hidden_dim, num_heads, ff_dim,
                 num_layers, num_classes, max_len=128, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)  # learned positions
        self.layers  = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, input_ids, mask=None):
        batch, seq = input_ids.shape
        pos = torch.arange(seq, device=input_ids.device).unsqueeze(0)
        
        # Embeddings
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(pos))
        
        # Encoder layers
        all_attn = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            all_attn.append(attn)
        
        # CLS pooling (first token) → classifier
        logits = self.classifier(x[:, 0, :])
        return logits, all_attn


# ✅ Quick test
m = CustomTransformerNLI(vocab_size=30522, hidden_dim=128, num_heads=4,
                         ff_dim=512, num_layers=2, num_classes=3)
ids   = torch.randint(0, 1000, (2, 20))
logits, attns = m(ids)
print(f"Logits shape: {logits.shape}  (batch=2, classes=3) ✅")
print(f"Attention layers: {len(attns)}, each: {attns[0].shape}")

---
## 🔧 Part 4: Prepare the XNLI Dataset

In [ ]:
# We use XLM-RoBERTa tokenizer — perfect for multilingual data!
# (handles all 15 languages in the dataset)
MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = 128

class XNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["premise"]),
            str(row["hypothesis"]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(row["label"], dtype=torch.long)
        }
        return item


# Train/val split from the training CSV
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['label']
)

# Use a subset for faster training (increase for better accuracy)
TRAIN_SIZE = 2000
VAL_SIZE   = 400

train_dataset = XNLIDataset(train_data.head(TRAIN_SIZE), tokenizer, MAX_LEN)
val_dataset   = XNLIDataset(val_data.head(VAL_SIZE),   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)}  | Val batches: {len(val_loader)}")

---
## 🔧 Part 5: Train Custom Transformer

In [ ]:
def train_custom(model, train_loader, val_loader, epochs=3, lr=1e-3):
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    history = {"train_loss": [], "val_acc": []}
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            logits, _ = model(ids, mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # --- Validate ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                ids    = batch["input_ids"].to(device)
                mask   = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                logits, _ = model(ids, mask)
                correct += (logits.argmax(-1) == labels).sum().item()
                total   += labels.size(0)
        
        avg_loss = total_loss / len(train_loader)
        val_acc  = correct / total
        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    return history


custom_model = CustomTransformerNLI(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=128,
    num_heads=4,
    ff_dim=512,
    num_layers=2,
    num_classes=3,
    max_len=MAX_LEN,
    dropout=0.1
)

print(f"Custom model params: {sum(p.numel() for p in custom_model.parameters()):,}")
print("\nTraining Custom Transformer...")
custom_history = train_custom(custom_model, train_loader, val_loader, epochs=3)

---
## 🔧 Part 6: Pretrained Multilingual DistilBERT Baseline

In [ ]:
def train_pretrained(model, train_loader, val_loader, epochs=2, lr=2e-5):
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    history = {"train_loss": [], "val_acc": []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                ids    = batch["input_ids"].to(device)
                mask   = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                out    = model(input_ids=ids, attention_mask=mask)
                correct += (out.logits.argmax(-1) == labels).sum().item()
                total   += labels.size(0)
        
        avg_loss = total_loss / len(train_loader)
        val_acc  = correct / total
        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    return history


pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
)

print(f"DistilBERT params: {sum(p.numel() for p in pretrained_model.parameters()):,}")
print("\nFine-tuning multilingual DistilBERT...")
pretrained_history = train_pretrained(pretrained_model, train_loader, val_loader, epochs=2)

---
## 📊 Part 7: Attention Map Visualization

In [ ]:
def visualize_attention(model, tokenizer, premise, hypothesis, layer=0, max_tokens=20):
    """Plot attention heatmaps for all heads in a given encoder layer."""
    enc = tokenizer(
        premise, hypothesis,
        max_length=MAX_LEN, truncation=True, return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(device)
    mask      = enc["attention_mask"].to(device)
    tokens    = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    model.eval()
    with torch.no_grad():
        logits, all_attn = model(input_ids, mask)
    
    # Clip to max_tokens for readability
    n     = min(max_tokens, len(tokens))
    attn  = all_attn[layer][0, :, :n, :n].cpu().numpy()  # (heads, n, n)
    toks  = tokens[:n]
    heads = attn.shape[0]
    pred  = label_names[logits.argmax(-1).item()]
    
    fig, axes = plt.subplots(1, heads, figsize=(5 * heads, 5))
    if heads == 1:
        axes = [axes]
    
    for h, ax in enumerate(axes):
        sns.heatmap(
            attn[h], ax=ax,
            xticklabels=toks, yticklabels=toks,
            cmap="Blues", linewidths=0.3, cbar=False
        )
        ax.set_title(f"Head {h+1}", fontsize=11)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', labelsize=7)
    
    fig.suptitle(
        f"Layer {layer+1} Attention — Prediction: {pred}\n"
        f"P: {premise[:60]}\nH: {hypothesis[:60]}",
        fontsize=10, y=1.05
    )
    plt.tight_layout()
    plt.show()


# Example 1 — English entailment
custom_model.to(device)
visualize_attention(
    custom_model, tokenizer,
    premise="A man is playing guitar on a stage.",
    hypothesis="A person is performing music.",
    layer=0
)

# Example 2 — English contradiction
visualize_attention(
    custom_model, tokenizer,
    premise="The cat is sleeping on the mat.",
    hypothesis="The cat is running outside.",
    layer=1
)

---
## 📊 Part 8: Results Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(custom_history["train_loss"],    marker='o', label="Custom Transformer",  color='steelblue')
axes[0].plot(pretrained_history["train_loss"], marker='s', label="mDistilBERT (fine-tuned)", color='coral')
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.4)

axes[1].plot(custom_history["val_acc"],    marker='o', label="Custom Transformer",  color='steelblue')
axes[1].plot(pretrained_history["val_acc"], marker='s', label="mDistilBERT (fine-tuned)", color='coral')
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.suptitle("Custom Transformer vs. Pretrained Multilingual DistilBERT on XNLI",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final Custom Transformer Val Acc:  {custom_history['val_acc'][-1]:.4f}")
print(f"Final mDistilBERT Val Acc:         {pretrained_history['val_acc'][-1]:.4f}")
print(f"\nCustom model params:   {sum(p.numel() for p in custom_model.parameters()):,}")
print(f"mDistilBERT params:    {sum(p.numel() for p in pretrained_model.parameters()):,}")

---
## 📝 Part 9: Reflection

### Single-Head vs Multi-Head Attention

| | Single-Head | Multi-Head |
|---|---|---|
| Attention types | 1 relationship at a time | Multiple in parallel |
| Subspace | Full hidden_dim | Smaller head_dim per head |
| Expressiveness | Lower | Higher |
| Compute | Lighter | Heavier |

Multi-head attention allows different heads to specialize — for example in this XNLI task one head might learn to align *subject* across premise/hypothesis while another tracks *negation* words that signal contradiction.

### Cross-Attention (used in encoder-decoder models)
In models like T5 or MarianMT for translation, the decoder uses **cross-attention**: Q comes from the decoder, but K and V come from the encoder. This lets the decoder "look at" the source sentence when generating each output word — essential for translation tasks like EN→ES.

### Custom Transformer vs Pretrained Multilingual BERT

| | Custom Transformer | mDistilBERT |
|---|---|---|
| Parameters | ~1-5M | ~134M |
| Pretrained on multilingual text | ❌ | ✅ |
| Handles 15 languages | Partially (via tokenizer) | Natively |
| Accuracy (small data) | Lower | Much higher |
| Training speed | Fast | Slower |
| Interpretability | Full control | Black box |

### Key Takeaway
The multilingual DistilBERT wins easily because it was pretrained on text from all 15 languages in the XNLI dataset. Our custom model starts from random weights and has no language knowledge — it needs far more data to compete. However, understanding how to build attention from scratch is essential for knowing *why* pretrained models work so well.